In [7]:
import requests
import json


def get_latest_article() -> dict:
    """Lấy bài viết mới được tạo gần đây nhất trên Wikipedia."""
    url = "https://en.wikipedia.org/w/api.php"

    headers = {
        # Thay bằng thông tin liên hệ thật.
        "User-Agent": "MyWikiFetcher/1.0 (your_email@example.com)"
    }

    def query(params: dict) -> dict:
        response = requests.get(
            url,
            headers=headers,
            params={
                "action": "query",
                "format": "json",
                "formatversion": 2,
                **params,
            },
            timeout=30,
        )
        response.raise_for_status()
        data = response.json()

        if "error" in data:
            raise RuntimeError(data["error"])

        return data["query"]

    # Lấy bài mới tạo, thuộc namespace bài viết, bỏ trang chuyển hướng.
    data = query({
        "list": "recentchanges",
        "rctype": "new",
        "rcnamespace": 0,
        "rcshow": "!redirect",
        "rcprop": "ids|title|timestamp",
        "rcdir": "older",
        "rclimit": 1,
    })

    changes = data["recentchanges"]
    if not changes:
        return {}

    article_id = changes[0]["pageid"]

    # Lấy nội dung văn bản và thời điểm sửa đổi gần nhất.
    data = query({
        "pageids": article_id,
        "prop": "extracts|revisions",
        "explaintext": 1,
        "rvprop": "timestamp",
        "rvlimit": 1,
        "rvdir": "older",
    })

    page = data["pages"][0]
    if page.get("missing", False):
        return {}

    # Lấy thời điểm của phiên bản đầu tiên.
    first_revision = query({
        "pageids": article_id,
        "prop": "revisions",
        "rvprop": "timestamp",
        "rvlimit": 1,
        "rvdir": "newer",
    })

    return {
        "article_id": article_id,
        "created_date": first_revision["pages"][0]["revisions"][0]["timestamp"],
        "updated_date": page["revisions"][0]["timestamp"],
        "title": page["title"],
        "content": page.get("extract", ""),
        "path": None,
    }


article = get_latest_article()
print(json.dumps(article, ensure_ascii=False, indent=2))

{
  "article_id": 84599592,
  "created_date": "2026-10-23T14:05:35Z",
  "updated_date": "2026-10-23T14:05:35Z",
  "title": "Agnes Hegedüs",
  "content": "Agnes Hegedüs (born 15 September 1964) is a Hungarian new media artist who works with video and interactive computer installations. Her works include The Fruit Machine (1991), Handsight (1992), Between the Words (1995) and Memory Theater VR (1997).\n\n\n== Education and career ==\nHegedüs was born in Budapest, Hungary. From 1985 to 1988, she studied photography, graphic design and video art at the Hungarian Academy of Applied Arts in Budapest. She continued her studies at the Minerva Academy in Groningen in 1988 and at the AKI Academy of Art in Enschede from 1988 to 1990. In 1990–1991, she undertook postgraduate studies at the Institute for New Media at the Städelschule in Frankfurt am Main, where she studied with Peter Weibel.\nIn 1992, she was an artist-in-residence at the ZKM Center for Art and Media Karlsruhe.\n\n\n== Artistic wor

In [8]:
print(article["content"])

Agnes Hegedüs (born 15 September 1964) is a Hungarian new media artist who works with video and interactive computer installations. Her works include The Fruit Machine (1991), Handsight (1992), Between the Words (1995) and Memory Theater VR (1997).


== Education and career ==
Hegedüs was born in Budapest, Hungary. From 1985 to 1988, she studied photography, graphic design and video art at the Hungarian Academy of Applied Arts in Budapest. She continued her studies at the Minerva Academy in Groningen in 1988 and at the AKI Academy of Art in Enschede from 1988 to 1990. In 1990–1991, she undertook postgraduate studies at the Institute for New Media at the Städelschule in Frankfurt am Main, where she studied with Peter Weibel.
In 1992, she was an artist-in-residence at the ZKM Center for Art and Media Karlsruhe.


== Artistic work ==
Hegedüs began making video works in the late 1980s. Her early works include Hierosgamos (1987), Bubble Order (1988) and And Grind Hard Stones to Meal (1989).